# GAN-B Baseline: Linux Training Notebook (Lambda Labs)

Trains a dual-head GAN-B model **without severity conditioning**.

- **Input:** synthetic healthy RGB (pre-generated and paired to diseased targets)
- **Outputs:** diseased RGB + lesion mask logits
- **Discriminator input:** concatenated [input RGB, output RGB] => 6 channels
- **Losses:** adversarial + L1 + SSIM + perceptual + BCE(mask) + Dice(mask)

Linux/cloud compatible with headless plotting and multiprocess dataloading.

In [2]:
# 1. Imports
import os
import sys
!{sys.executable} -m pip install tqdm
from tqdm import tqdm
import json
import time
import torch
import random
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib
matplotlib.use('Agg')  # headless backend for cloud/SSH
import matplotlib.pyplot as plt
import numpy as np

Defaulting to user installation because normal site-packages is not writeable


In [4]:
# 2. Import models and losses
from pathlib import Path
import importlib

# Resolve repo root (notebook is at src/gan/cyclegan_train_linux.ipynb)
repo_root = Path.cwd().resolve()
if repo_root.name == 'gan':
    repo_root = repo_root.parents[1]
elif repo_root.name == 'src':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f'Repo root: {repo_root}')

import src.models.unet_ganb_dualhead as ganb_unet_mod
import src.models.patchgan_discriminator as disc_mod
import src.models.losses as loss_mod
importlib.reload(ganb_unet_mod)
importlib.reload(disc_mod)
importlib.reload(loss_mod)

UNetDualHead = ganb_unet_mod.UNetDualHead
PatchGANDiscriminator = disc_mod.PatchGANDiscriminator
GANLoss = loss_mod.GANLoss
PerceptualLoss = loss_mod.PerceptualLoss
SSIMLoss = loss_mod.SSIMLoss
print('GAN-B modules loaded.')

Repo root: /lambda/nfs/test-2
GAN-B modules loaded.


In [15]:
# 3. Configuration

class Config:
    seed: int = 42
    image_size: int = 512
    batch_size: int = 4
    num_workers: int = 4
    num_epochs: int = 150
    train_split: float = 0.9

    g_lr: float = 1e-4
    d_lr: float = 2e-4
    beta1: float = 0.5
    beta2: float = 0.999

    lambda_adv: float = 1.0
    lambda_l1: float = 100.0
    lambda_ssim: float = 10.0
    lambda_perc: float = 10.0
    lambda_mask_bce: float = 25.0
    lambda_mask_dice: float = 25.0

    save_every: int = 10
    sample_every: int = 5
    run_name: str = 'ganB_baseline_no_severity_final'

CFG = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

HEALTHY_DIR = repo_root / 'data' / 'ganb' / 'healthy'
DISEASED_DIR = repo_root / 'data' / 'ganb' / 'image'
MASK_DIR = repo_root / 'data' / 'ganb' / 'mask'
CHECKPOINT_DIR = repo_root / 'src' / 'models' / 'checkpoints'
LOG_DIR = repo_root / 'results' / 'logs'
SAMPLE_DIR = repo_root / 'results' / 'figures' / 'training_samples' / 'ganB_baseline'

for p in [CHECKPOINT_DIR, LOG_DIR, SAMPLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(
    f'Data dirs:\n'
    f'- healthy: {HEALTHY_DIR}\n'
    f'- diseased: {DISEASED_DIR}\n'
    f'- mask: {MASK_DIR}'
)

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
Data dirs:
- healthy: /lambda/nfs/test-2/data/ganb/healthy
- diseased: /lambda/nfs/test-2/data/ganb/image
- mask: /lambda/nfs/test-2/data/ganb/mask


In [16]:
# 4. Dataset + pairing audit (pre-generated healthy inputs)
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def parse_leaf_index_from_image(name: str):
    stem = Path(name).stem
    if stem.startswith('leaf'):
        token = stem[4:]
        if token.isdigit():
            return int(token)
    return None


def parse_leaf_index_from_mask(name: str):
    stem = Path(name).stem
    if stem.startswith('mask_leaf'):
        token = stem[len('mask_leaf'):]
        if token.isdigit():
            return int(token)
    return None


def index_files_by_leaf(files, parser):
    out = {}
    for f in files:
        idx = parser(f)
        if idx is not None:
            out[idx] = f
    return out


def build_triplets_and_audit(healthy_dir: Path, diseased_dir: Path, mask_dir: Path):
    exts = ('.png', '.jpg', '.jpeg')
    healthy_files = sorted([f for f in os.listdir(healthy_dir) if f.lower().endswith(exts)])
    diseased_files = sorted([f for f in os.listdir(diseased_dir) if f.lower().endswith(exts)])
    mask_files = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith(exts)])

    healthy_by_idx = index_files_by_leaf(healthy_files, parse_leaf_index_from_image)
    diseased_by_idx = index_files_by_leaf(diseased_files, parse_leaf_index_from_image)
    masks_by_idx = index_files_by_leaf(mask_files, parse_leaf_index_from_mask)

    common = sorted(set(healthy_by_idx).intersection(diseased_by_idx).intersection(masks_by_idx))
    only_healthy = sorted(set(healthy_by_idx) - set(common))
    only_diseased = sorted(set(diseased_by_idx) - set(common))
    only_masks = sorted(set(masks_by_idx) - set(common))

    triplets = [
        (
            healthy_dir / healthy_by_idx[i],
            diseased_dir / diseased_by_idx[i],
            mask_dir / masks_by_idx[i],
            diseased_by_idx[i],
        )
        for i in common
    ]

    audit = {
        'n_healthy_files': len(healthy_files),
        'n_diseased_files': len(diseased_files),
        'n_mask_files': len(mask_files),
        'n_parsed_healthy': len(healthy_by_idx),
        'n_parsed_diseased': len(diseased_by_idx),
        'n_parsed_masks': len(masks_by_idx),
        'n_triplets': len(triplets),
        'only_healthy_indices': only_healthy[:20],
        'only_diseased_indices': only_diseased[:20],
        'only_mask_indices': only_masks[:20],
    }
    return triplets, audit


def split_triplets(triplets, train_split: float, seed: int):
    indices = list(range(len(triplets)))
    rng = random.Random(seed)
    rng.shuffle(indices)
    n_train = int(len(indices) * train_split)
    train_idx = indices[:n_train]
    val_idx = indices[n_train:]
    train_triplets = [triplets[i] for i in train_idx]
    val_triplets = [triplets[i] for i in val_idx]
    return train_triplets, val_triplets


class DirectGanBDataset(Dataset):
    """Loads pre-generated healthy input + diseased target + lesion mask."""

    def __init__(self, triplets, image_size: int):
        self.triplets = triplets
        self.rgb_tfm = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])
        self.mask_tfm = transforms.Compose([
            transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        healthy_path, diseased_path, mask_path, fname = self.triplets[idx]
        healthy_pil = Image.open(healthy_path).convert('RGB')
        diseased_pil = Image.open(diseased_path).convert('RGB')
        mask_pil = Image.open(mask_path).convert('L')

        mask_np = np.array(mask_pil)
        mask_bin = (mask_np > 127).astype(np.uint8) * 255

        input_t = self.rgb_tfm(healthy_pil)
        target_img_t = self.rgb_tfm(diseased_pil)
        target_mask_t = self.mask_tfm(Image.fromarray(mask_bin)).clamp(0, 1)
        return input_t, target_img_t, target_mask_t, fname


set_seed(CFG.seed)
triplets, audit = build_triplets_and_audit(HEALTHY_DIR, DISEASED_DIR, MASK_DIR)
print(json.dumps(audit, indent=2))

audit_path = LOG_DIR / f'{CFG.run_name}_dataset_audit.json'
with open(audit_path, 'w', encoding='utf-8') as f:
    json.dump(audit, f, indent=2)
print(f'Audit saved: {audit_path}')

train_triplets, val_triplets = split_triplets(triplets, train_split=CFG.train_split, seed=CFG.seed)
print(f'Train triplets: {len(train_triplets)} | Val triplets: {len(val_triplets)}')

train_ds = DirectGanBDataset(train_triplets, image_size=CFG.image_size)
val_ds = DirectGanBDataset(val_triplets, image_size=CFG.image_size)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)

{
  "n_healthy_files": 263,
  "n_diseased_files": 263,
  "n_mask_files": 263,
  "n_parsed_healthy": 263,
  "n_parsed_diseased": 263,
  "n_parsed_masks": 263,
  "n_triplets": 263,
  "only_healthy_indices": [],
  "only_diseased_indices": [],
  "only_mask_indices": []
}
Audit saved: /lambda/nfs/test-2/results/logs/ganB_baseline_no_severity_final_dataset_audit.json
Train triplets: 236 | Val triplets: 27


In [17]:
# 5. Models, losses, optimizers
net_g = UNetDualHead(in_channels=3, image_out_channels=3, mask_out_channels=1).to(DEVICE)
net_d = PatchGANDiscriminator(in_channels=6, base_features=64, n_layers=3).to(DEVICE)

criterion_gan = GANLoss(mode='lsgan').to(DEVICE)
criterion_l1 = nn.L1Loss().to(DEVICE)
criterion_ssim = SSIMLoss().to(DEVICE)
criterion_perc = PerceptualLoss().to(DEVICE)
criterion_bce_logits = nn.BCEWithLogitsLoss().to(DEVICE)

opt_g = optim.Adam(net_g.parameters(), lr=CFG.g_lr, betas=(CFG.beta1, CFG.beta2))
opt_d = optim.Adam(net_d.parameters(), lr=CFG.d_lr, betas=(CFG.beta1, CFG.beta2))

scaler_g = GradScaler('cuda' if DEVICE.type == 'cuda' else 'cpu')
scaler_d = GradScaler('cuda' if DEVICE.type == 'cuda' else 'cpu')

g_params = sum(p.numel() for p in net_g.parameters() if p.requires_grad)
d_params = sum(p.numel() for p in net_d.parameters() if p.requires_grad)
print(f'Generator params: {g_params:,}')
print(f'Discriminator params: {d_params:,}')

Generator params: 31,043,716
Discriminator params: 2,767,809


In [18]:
# 6. Helpers: dice loss, sampling, evaluation
def dice_loss_from_logits(mask_logits, target_mask, eps=1e-6):
    pred = torch.sigmoid(mask_logits)
    pred_flat = pred.view(pred.shape[0], -1)
    target_flat = target_mask.view(target_mask.shape[0], -1)
    inter = (pred_flat * target_flat).sum(dim=1)
    denom = pred_flat.sum(dim=1) + target_flat.sum(dim=1)
    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()

def save_sample_panel(epoch, batch, input_t, target_img_t, target_mask_t, gen_img_t, gen_mask_logits_t):
    def to_np(t):
        return t[0].detach().cpu().float().permute(1, 2, 0).numpy().clip(0, 1).astype(np.float32)

    in_np    = to_np(input_t)
    tgt_np   = to_np(target_img_t)
    g_np     = to_np(gen_img_t)
    tgt_mask  = target_mask_t[0, 0].detach().cpu().float().numpy().astype(np.float32)
    pred_mask = torch.sigmoid(gen_mask_logits_t[0, 0]).detach().cpu().float().numpy().astype(np.float32)

    fig, axs = plt.subplots(1, 5, figsize=(24, 5))
    axs[0].imshow(in_np);                              axs[0].set_title('Input (Synthetic Healthy)'); axs[0].axis('off')
    axs[1].imshow(g_np);                               axs[1].set_title('Generated Diseased');        axs[1].axis('off')
    axs[2].imshow(tgt_np);                             axs[2].set_title('Target Diseased');           axs[2].axis('off')
    axs[3].imshow(pred_mask, cmap='gray', vmin=0, vmax=1); axs[3].set_title('Predicted Mask');       axs[3].axis('off')
    axs[4].imshow(tgt_mask,  cmap='gray', vmin=0, vmax=1); axs[4].set_title('Target Mask');          axs[4].axis('off')

    plt.tight_layout()
    out_path = SAMPLE_DIR / f'{CFG.run_name}_ep{epoch:03d}_b{batch:04d}.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

def evaluate(model_g, loader):
    model_g.eval()
    total_l1 = 0.0
    total_ssim = 0.0
    total_mask_bce = 0.0
    total_mask_dice = 0.0
    n = 0
    with torch.no_grad():
        for inp, target_img, target_mask, _ in loader:
            inp = inp.to(DEVICE, non_blocking=True)
            target_img = target_img.to(DEVICE, non_blocking=True)
            target_mask = target_mask.to(DEVICE, non_blocking=True)
            out_img, out_mask_logits = model_g(inp)
            total_l1 += criterion_l1(out_img, target_img).item()
            total_ssim += criterion_ssim.ssim_value(out_img, target_img).item()
            total_mask_bce += criterion_bce_logits(out_mask_logits, target_mask).item()
            total_mask_dice += dice_loss_from_logits(out_mask_logits, target_mask).item()
            n += 1
    if n == 0:
        return {'l1': 0.0, 'ssim': 0.0, 'mask_bce': 0.0, 'mask_dice': 0.0}
    return {
        'l1': total_l1 / n,
        'ssim': total_ssim / n,
        'mask_bce': total_mask_bce / n,
        'mask_dice': total_mask_dice / n,
    }

In [19]:
# 7. Training loop
history = []
print(f'Training for {CFG.num_epochs} epochs...')

for epoch in range(1, CFG.num_epochs + 1):
    epoch_start = time.time()
    net_g.train()
    net_d.train()

    running = {
        'd': 0.0,
        'g': 0.0,
        'g_adv': 0.0,
        'g_l1': 0.0,
        'g_ssim': 0.0,
        'g_perc': 0.0,
        'g_mask_bce': 0.0,
        'g_mask_dice': 0.0,
    }

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{CFG.num_epochs}')
    for step, (inp, target_img, target_mask, _) in enumerate(pbar, start=1):
        inp = inp.to(DEVICE, non_blocking=True)
        target_img = target_img.to(DEVICE, non_blocking=True)
        target_mask = target_mask.to(DEVICE, non_blocking=True)

        # Train D
        opt_d.zero_grad(set_to_none=True)
        with torch.no_grad():
            fake_img_detach, _ = net_g(inp)

        with autocast('cuda' if DEVICE.type == 'cuda' else 'cpu'):
            real_pair = torch.cat([inp, target_img], dim=1)
            fake_pair = torch.cat([inp, fake_img_detach], dim=1)
            pred_real = net_d(real_pair)
            pred_fake = net_d(fake_pair)
            loss_d_real = criterion_gan(pred_real, True)
            loss_d_fake = criterion_gan(pred_fake, False)
            loss_d = 0.5 * (loss_d_real + loss_d_fake)

        scaler_d.scale(loss_d).backward()
        scaler_d.step(opt_d)
        scaler_d.update()

        # Train G
        opt_g.zero_grad(set_to_none=True)
        with autocast('cuda' if DEVICE.type == 'cuda' else 'cpu'):
            out_img, out_mask_logits = net_g(inp)
            pred_fake_for_g = net_d(torch.cat([inp, out_img], dim=1))
            loss_g_adv = criterion_gan(pred_fake_for_g, True)
            loss_g_l1 = criterion_l1(out_img, target_img)
            loss_g_ssim = criterion_ssim(out_img, target_img)
            loss_g_perc = criterion_perc(out_img, target_img)
            loss_g_mask_bce = criterion_bce_logits(out_mask_logits, target_mask)
            loss_g_mask_dice = dice_loss_from_logits(out_mask_logits, target_mask)
            loss_g = (
                CFG.lambda_adv * loss_g_adv
                + CFG.lambda_l1 * loss_g_l1
                + CFG.lambda_ssim * loss_g_ssim
                + CFG.lambda_perc * loss_g_perc
                + CFG.lambda_mask_bce * loss_g_mask_bce
                + CFG.lambda_mask_dice * loss_g_mask_dice
            )

        scaler_g.scale(loss_g).backward()
        scaler_g.step(opt_g)
        scaler_g.update()

        running['d'] += loss_d.item()
        running['g'] += loss_g.item()
        running['g_adv'] += loss_g_adv.item()
        running['g_l1'] += loss_g_l1.item()
        running['g_ssim'] += loss_g_ssim.item()
        running['g_perc'] += loss_g_perc.item()
        running['g_mask_bce'] += loss_g_mask_bce.item()
        running['g_mask_dice'] += loss_g_mask_dice.item()

        pbar.set_postfix({
            'D': f'{loss_d.item():.3f}',
            'G': f'{loss_g.item():.3f}',
            'L1': f'{loss_g_l1.item():.4f}',
            'Dice': f'{loss_g_mask_dice.item():.4f}',
        })

        if epoch % CFG.sample_every == 0 and step == 1:
            save_sample_panel(epoch, step, inp, target_img, target_mask, out_img, out_mask_logits)

    n_steps = max(len(train_loader), 1)
    train_log = {f'train_{k}': v / n_steps for k, v in running.items()}
    val_log = evaluate(net_g, val_loader)
    val_log = {f'val_{k}': v for k, v in val_log.items()}
    epoch_log = {'epoch': epoch, 'epoch_time_sec': time.time() - epoch_start, **train_log, **val_log}
    history.append(epoch_log)

    print(
        f"[Epoch {epoch}] D={epoch_log['train_d']:.4f} G={epoch_log['train_g']:.4f} "
        f"Val L1={epoch_log['val_l1']:.4f} Val SSIM={epoch_log['val_ssim']:.4f} "
        f"Val MaskDiceLoss={epoch_log['val_mask_dice']:.4f}"
    )

    if epoch % CFG.save_every == 0 or epoch == CFG.num_epochs:
        ckpt_path = CHECKPOINT_DIR / f'{CFG.run_name}_epoch{epoch}.pth'
        torch.save({
            'epoch': epoch,
            'config': CFG.__dict__,
            'netG_state_dict': net_g.state_dict(),
            'netD_state_dict': net_d.state_dict(),
            'optG_state_dict': opt_g.state_dict(),
            'optD_state_dict': opt_d.state_dict(),
            'history': history,
        }, ckpt_path)
        print(f'Checkpoint saved: {ckpt_path}')

history_json = LOG_DIR / f'{CFG.run_name}_history.json'
with open(history_json, 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2)
print(f'History json saved: {history_json}')

history_csv = LOG_DIR / f'{CFG.run_name}_history.csv'
with open(history_csv, 'w', newline='', encoding='utf-8') as f:
    if history:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)
print(f'History csv saved: {history_csv}')

Training for 150 epochs...


Epoch 1/150: 100%|██████████| 59/59 [00:10<00:00,  5.89it/s, D=0.229, G=64.033, L1=0.1476, Dice=0.6381]


[Epoch 1] D=1.2796 G=81.4280 Val L1=0.1736 Val SSIM=0.3836 Val MaskDiceLoss=0.6956


Epoch 2/150: 100%|██████████| 59/59 [00:09<00:00,  5.91it/s, D=0.074, G=55.048, L1=0.1140, Dice=0.5674]


[Epoch 2] D=0.1790 G=63.5728 Val L1=0.1474 Val SSIM=0.4112 Val MaskDiceLoss=0.6354


...

Epoch 149/150: 100%|██████████| 59/59 [00:09<00:00,  5.92it/s, D=0.076, G=5.366, L1=0.0086, Dice=0.0505] 


[Epoch 149] D=0.1466 G=8.4015 Val L1=0.0102 Val SSIM=0.9610 Val MaskDiceLoss=0.0715


Epoch 150/150: 100%|██████████| 59/59 [00:11<00:00,  5.20it/s, D=0.061, G=6.745, L1=0.0098, Dice=0.0659] 


[Epoch 150] D=0.1044 G=8.6054 Val L1=0.0092 Val SSIM=0.9627 Val MaskDiceLoss=0.0680
Checkpoint saved: /lambda/nfs/test-2/src/models/checkpoints/ganB_baseline_no_severity_final_epoch150.pth
History json saved: /lambda/nfs/test-2/results/logs/ganB_baseline_no_severity_final_history.json


NameError: name 'csv' is not defined

In [20]:
# 8. Plot training curves
if not history:
    raise RuntimeError('No history found. Run training cell first.')

epochs = [h['epoch'] for h in history]

fig, axs = plt.subplots(2, 3, figsize=(18, 10))
axs[0, 0].plot(epochs, [h['train_g'] for h in history], label='Train G')
axs[0, 0].set_title('Generator Total Loss')
axs[0, 0].grid(True)

axs[0, 1].plot(epochs, [h['train_d'] for h in history], label='Train D', color='r')
axs[0, 1].set_title('Discriminator Loss')
axs[0, 1].grid(True)

axs[0, 2].plot(epochs, [h['train_g_l1'] for h in history], label='Train L1', color='g')
axs[0, 2].plot(epochs, [h['val_l1'] for h in history], label='Val L1', color='g', linestyle='--')
axs[0, 2].set_title('L1 Loss')
axs[0, 2].legend()
axs[0, 2].grid(True)

axs[1, 0].plot(epochs, [h['train_g_ssim'] for h in history], label='Train SSIM loss', color='m')
axs[1, 0].set_title('SSIM Loss (1-SSIM)')
axs[1, 0].grid(True)

axs[1, 1].plot(epochs, [h['train_g_mask_bce'] for h in history], label='Train BCE', color='c')
axs[1, 1].plot(epochs, [h['train_g_mask_dice'] for h in history], label='Train Dice', color='k')
axs[1, 1].plot(epochs, [h['val_mask_dice'] for h in history], label='Val Dice', color='k', linestyle='--')
axs[1, 1].set_title('Mask Losses')
axs[1, 1].legend()
axs[1, 1].grid(True)

axs[1, 2].plot(epochs, [h['val_ssim'] for h in history], label='Val SSIM', color='orange')
axs[1, 2].set_title('Validation SSIM Value')
axs[1, 2].grid(True)

for ax in axs.flatten():
    ax.set_xlabel('Epoch')

plt.suptitle('GAN-B Baseline (No Severity Conditioning)')
plt.tight_layout()
curve_path = LOG_DIR / f'{CFG.run_name}_curves.png'
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Curves saved: {curve_path}')

Curves saved: /lambda/nfs/test-2/results/logs/ganB_baseline_no_severity_final_curves.png


In [21]:
# 9. Full validation metrics summary
metrics = evaluate(net_g, val_loader)
print('=' * 50)
print('GAN-B VALIDATION SUMMARY')
print('=' * 50)
print(f"Val L1:         {metrics['l1']:.6f}")
print(f"Val SSIM:       {metrics['ssim']:.6f}")
print(f"Val Mask BCE:   {metrics['mask_bce']:.6f}")
print(f"Val Mask DiceL: {metrics['mask_dice']:.6f}")
print('=' * 50)

GAN-B VALIDATION SUMMARY
Val L1:         0.009193
Val SSIM:       0.962723
Val Mask BCE:   0.040859
Val Mask DiceL: 0.068019
